# F02 차례 ①·③·⑤·⑥ 검수 — v2.1

입력: 원고 1개 + 선택적인 개발계획서 1개 (`.hwp` / `.hwpx`, 순서 무관, 내용으로 역할 판별).
출력: 항목별 JSON, 요약, 지적사항, 전체 결과, 진단 정보, ZIP.

**② 행동명사 · ④ 교수/평가 위치 · ⑦ 쪽수는 미구현입니다.**
이 결과만으로 차례 7개 항목 전체의 적합 여부를 판단하지 마세요.

### 판정
- 개발계획서가 없으면 ①의 대조는 생략하고 ③은 차례 기준으로 비교합니다(△).
- ⑥ 본문 존재가 불확실한 경우 삭제를 단정하지 않습니다.

### 처리 흐름
파일 읽기 → 차례/개요/본문/계획서 추출 → ①③⑤⑥ 판정 → JSON/ZIP 저장.
기본 양식은 section0=차례·개요, section1=본문입니다. 아래 설정에서 변경할 수 있습니다.

위에서 아래로 실행하세요. Python 3.10 이상이며 추가 설치 패키지는 없습니다.
이 노트북은 `tools/build_notebook.py`로 Python 패키지에서 생성한 배포본입니다.
팀의 수정 원본은 `src/ncs_core/`(공통)와 `src/ncs_toc/`이고, 수정 후 노트북을 다시 생성합니다.

- `O`: 충족, `X`: 위반, `△`: 사람 확인, `-`: 해당 없음 또는 대조 생략.
- 행 판정(검수결과): X가 하나라도 있으면 `부적합`, X 없이 △가 있으면 `판단불가`, 나머지는 `적합`.
- `*_00_요약.json`의 `검수결과`는 검수 확인서 표현대로 적합/부적합/판단불가로 종합한다(부적합 > 판단불가 > 적합).


## 1. 실행 환경 준비
임시 폴더에 이 배포본의 패키지를 구성합니다. 기존 프로젝트 파일은 덮어쓰지 않습니다.


In [ ]:
from pathlib import Path
import importlib
import json
import sys
import tempfile

RUNTIME_DIR = Path(tempfile.mkdtemp(prefix="ncs_toc_"))
# 실패하거나 재실행해도 이전 결과를 다시 쓰지 않도록 초기화합니다.
result = saved = None
PATHS = []


def write_module(relative_path, source):
    path = RUNTIME_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")


print("검수 엔진 위치:", RUNTIME_DIR)


## 2. 검수 엔진
각 셀은 한 모듈의 소스입니다(접혀 있음). 사용만 할 때는 그대로 실행하세요.


In [ ]:
# 모듈: ncs_core/__init__.py
write_module('ncs_core/__init__.py', r'''"""F02 검수 공통 코어 (Python 3.10+, 추가 설치 패키지 없음).

두 검수 패키지(ncs_toc, ncs_position)가 공유하는 부분:
문서 읽기, 문자 정규화·판정, 입력 역할 판별, 모듈 코드, 규칙 실행, 결과 저장.
검수 정책(무엇을 X로 볼지)은 여기에 두지 않는다.
"""

from .documents import Document, Table, load_document
from .inputs import detect_role, select_documents, validate_distinct
from .models import ReviewResult, SavedReport
from .module_code import resolve_module_code, validate_module_code
from .reporting import save_report
from .rules import Rule, run_rules
from .text import FAIL, NA, O, PASS, PUA, TRI, UNDECIDED, VERDICTS, X, flatten, make_normalizer, verdict, visible

__all__ = [
    "Document",
    "Table",
    "load_document",
    "detect_role",
    "select_documents",
    "validate_distinct",
    "ReviewResult",
    "SavedReport",
    "resolve_module_code",
    "validate_module_code",
    "save_report",
    "Rule",
    "run_rules",
    "NA",
    "PASS",
    "FAIL",
    "UNDECIDED",
    "VERDICTS",
    "O",
    "PUA",
    "TRI",
    "X",
    "flatten",
    "make_normalizer",
    "verdict",
    "visible",
]
__version__ = "1.0.0"
''')


In [ ]:
# 모듈: ncs_core/documents.py
write_module('ncs_core/documents.py', r'''"""문서를 한 번 읽어 Document로 만든다. 검수 정책이나 Colab에 의존하지 않는다."""

import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from . import hwp, hwpx

Table = dict[str, Any]


@dataclass(frozen=True)
class Document:
    """읽은 문서. 모든 텍스트는 연속 공백이 하나로 정리된 상태다.

    sections: 섹션별 문단 텍스트(표 셀 안 문단 포함, 문서 순서).
    tables:   표 목록(문서 순서). 'styles'는 hwpx일 때만 채워진다.
    images:   (섹션, 문단 인덱스, 원본 그림 이름). hwpx만.
    """

    path: Path
    format: str
    sections: list[list[str]]
    tables: list[Table] = field(default_factory=list)
    images: list[tuple[int, int, str]] = field(default_factory=list)

    @property
    def has_styles(self) -> bool:
        return self.format == "hwpx"

    def section_tables(self, index: int) -> list[Table]:
        return [t for t in self.tables if t["section"] == index]


def detect_format(path) -> str:
    """파일 내용으로 hwp/hwpx를 판별한다. 둘 다 아니면 ValueError."""
    path = Path(path)
    with path.open("rb") as stream:
        head = stream.read(8)
    if head == hwp.OLE_MAGIC:
        if not hwp.is_hwp(path.read_bytes()):
            raise ValueError(f"OLE 파일이지만 HWP 서명이 아님: {path}")
        return "hwp"
    if head[:2] == b"PK":
        if hwpx.is_hwpx(path):
            return "hwpx"
        raise ValueError(f"zip이지만 hwpx가 아님: {path}")
    raise ValueError(f"hwp/hwpx 둘 다 아님 (머리 바이트 {head!r}): {path}")


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def load_document(path) -> Document:
    path = Path(path)
    fmt = detect_format(path)
    if fmt == "hwp":
        sections, tables = hwp.load_hwp(path.read_bytes())
        images = []
    else:
        sections, tables, images = hwpx.load_hwpx(path)
    sections = [[_clean(p) for p in section] for section in sections]
    for table in tables:
        table["cells"] = {key: _clean(value) for key, value in table["cells"].items()}
    if not any(line for section in sections for line in section):
        raise ValueError(f"문단 텍스트를 추출하지 못했습니다: {path}")
    return Document(path, fmt, sections, tables, images)
''')


In [ ]:
# 모듈: ncs_core/hwp.py
write_module('ncs_core/hwp.py', r'''"""HWP(바이너리, OLE/CFB) 읽기. 문단 텍스트와 표 셀 텍스트만 읽고 서식은 읽지 않는다."""

import struct
import zlib
from collections.abc import Iterator

OLE_MAGIC = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"
HWP_SIGNATURE = b"HWP Document File"
END_OF_CHAIN = (0xFFFFFFFE, 0xFFFFFFFF)

TAG_PARA_TEXT, TAG_LIST_HEADER, TAG_TABLE = 0x43, 0x48, 0x4D
# 문단 텍스트의 제어 문자 중 2바이트짜리. 나머지 제어 문자는 16바이트.
SHORT_CONTROL_CHARS = (0, 10, 13, 24, 25, 26, 27, 28, 29, 30, 31)


def read_cfb(data: bytes):
    """CFB 컨테이너를 열어 (read_stream, 스트림 이름 목록)을 돌려준다.

    섹터 n은 파일 오프셋 (n+1)*섹터크기에서 시작한다(CFB 규격).
    순환하는 섹터 체인은 손상으로 보고 예외를 낸다.
    """
    header = data[:512]
    sector_size = 1 << struct.unpack_from("<H", header, 0x1E)[0]
    mini_sector_size = 1 << struct.unpack_from("<H", header, 0x20)[0]
    dir_start = struct.unpack_from("<I", header, 0x30)[0]
    mini_cutoff = struct.unpack_from("<I", header, 0x38)[0]
    minifat_start = struct.unpack_from("<I", header, 0x3C)[0]
    difat_start = struct.unpack_from("<I", header, 0x44)[0]
    difat = list(struct.unpack_from("<109I", header, 0x4C))
    entries_per_sector = sector_size // 4

    def sector(n: int) -> bytes:
        return data[(n + 1) * sector_size : (n + 2) * sector_size]

    next_difat, seen = difat_start, set()
    while next_difat not in END_OF_CHAIN:
        if next_difat in seen:
            raise ValueError("손상된 HWP: DIFAT 순환 참조")
        seen.add(next_difat)
        values = struct.unpack(f"<{entries_per_sector}I", sector(next_difat))
        difat += values[:-1]
        next_difat = values[-1]

    fat: list[int] = []
    for fat_sector in difat:
        if fat_sector not in END_OF_CHAIN:
            fat += struct.unpack(f"<{entries_per_sector}I", sector(fat_sector))

    def chain(start: int, table: list[int]) -> list[int]:
        out, current, visited = [], start, set()
        while current not in END_OF_CHAIN:
            if current in visited or current >= len(table):
                raise ValueError("손상된 HWP: 잘못된 섹터 체인")
            visited.add(current)
            out.append(current)
            current = table[current]
        return out

    def read_chain(start: int, size: int) -> bytes:
        return b"".join(sector(s) for s in chain(start, fat))[:size]

    directory = b"".join(sector(s) for s in chain(dir_start, fat))
    entries = []
    for i in range(len(directory) // 128):
        entry = directory[i * 128 : (i + 1) * 128]
        name_length = struct.unpack_from("<H", entry, 0x40)[0]
        entries.append(
            (
                entry[: max(name_length - 2, 0)].decode("utf-16le", "ignore"),
                entry[0x42],
                struct.unpack_from("<I", entry, 0x74)[0],
                struct.unpack_from("<Q", entry, 0x78)[0],
            )
        )
    ministream = read_chain(entries[0][2], entries[0][3])
    minifat: list[int] = []
    for s in chain(minifat_start, fat):
        minifat += struct.unpack(f"<{entries_per_sector}I", sector(s))

    def read_stream(name: str) -> bytes | None:
        for entry_name, kind, start, size in entries:
            if entry_name == name and kind == 2:
                if size < mini_cutoff:
                    return b"".join(
                        ministream[s * mini_sector_size : (s + 1) * mini_sector_size]
                        for s in chain(start, minifat)
                    )[:size]
                return read_chain(start, size)
        return None

    return read_stream, [name for name, kind, *_ in entries if kind == 2]


def records(buffer: bytes) -> Iterator[tuple[int, int, bytes]]:
    """본문 스트림을 (tag, level, payload)로 나눈다. 잘린 레코드는 예외."""
    i = 0
    while i + 4 <= len(buffer):
        head = struct.unpack_from("<I", buffer, i)[0]
        i += 4
        tag, level, size = head & 0x3FF, (head >> 10) & 0x3FF, head >> 20
        if size == 0xFFF:
            if i + 4 > len(buffer):
                raise ValueError("손상된 HWP: 확장 레코드 길이 누락")
            size = struct.unpack_from("<I", buffer, i)[0]
            i += 4
        if i + size > len(buffer):
            raise ValueError("손상된 HWP: 레코드 잘림")
        yield tag, level, buffer[i : i + size]
        i += size


def para_text(payload: bytes) -> str:
    """PARA_TEXT 레코드에서 문자열을 꺼낸다. 탭·줄바꿈은 공백으로."""
    out, i, n = [], 0, len(payload)
    while i + 2 <= n:
        code = struct.unpack_from("<H", payload, i)[0]
        if code < 32:
            i += 2 if code in SHORT_CONTROL_CHARS else 16
            if code in (9, 10, 13):
                out.append(" ")
            continue
        out.append(chr(code))
        i += 2
    return "".join(out)


def is_hwp(data: bytes) -> bool:
    """OLE 파일이고 FileHeader 스트림의 서명이 HWP인가."""
    if data[:8] != OLE_MAGIC:
        return False
    try:
        read_stream, _ = read_cfb(data)
        header = read_stream("FileHeader")
    except (ValueError, struct.error, IndexError):
        return False
    return header is not None and header[:32].rstrip(b"\x00") == HWP_SIGNATURE


def load_hwp(data: bytes) -> tuple[list[list[str]], list[dict]]:
    """(섹션별 문단 목록, 표 목록)을 돌려준다.

    문단 목록은 모든 PARA_TEXT 레코드(표 셀 안 포함)를 순서대로 담는다.
    표의 `paragraph`는 그 표를 가진 문단의 인덱스(같은 기준)다.
    """
    read_stream, names = read_cfb(data)
    file_header = read_stream("FileHeader")
    if file_header is None:
        raise ValueError("HWP FileHeader 스트림이 없음")
    flags = struct.unpack_from("<I", file_header, 36)[0]
    if flags & 2:
        raise ValueError("암호화된 hwp는 읽을 수 없음")
    section_names = sorted(
        (n for n in names if n.startswith("Section")), key=lambda s: int(s[7:])
    )
    sections, tables = [], []
    for section_index, name in enumerate(section_names):
        raw = read_stream(name)
        if flags & 1:
            raw = zlib.decompress(raw, -15)
        paragraphs: list[str] = []
        stack: list[tuple[int, dict]] = []  # (level, table)
        cell: tuple[int, dict, tuple[int, int]] | None = None
        for tag, level, payload in records(raw):
            if tag == TAG_PARA_TEXT:
                paragraphs.append(para_text(payload))
                if cell and level == cell[0] + 1:
                    table, key = cell[1], cell[2]
                    table["cells"][key] = (table["cells"][key] + " " + paragraphs[-1]).strip()
            elif tag == TAG_TABLE:
                rows, cols = struct.unpack_from("<HH", payload, 4)
                table = {
                    "section": section_index,
                    "paragraph": max(len(paragraphs) - 1, 0),
                    "rows": rows,
                    "cols": cols,
                    "cells": {},
                    "styles": None,
                }
                while stack and stack[-1][0] >= level:
                    stack.pop()
                stack.append((level, table))
                tables.append(table)
            elif tag == TAG_LIST_HEADER:
                while stack and stack[-1][0] > level:
                    stack.pop()  # 안쪽 표가 끝남
                if stack and stack[-1][0] == level and len(payload) >= 16:
                    col, row = struct.unpack_from("<HH", payload, 8)
                    table = stack[-1][1]
                    table["cells"].setdefault((row, col), "")
                    cell = (level, table, (row, col))
                else:
                    cell = None
        sections.append(paragraphs)
    return sections, tables
''')


In [ ]:
# 모듈: ncs_core/hwpx.py
write_module('ncs_core/hwpx.py', r'''"""HWPX(zip+XML) 읽기. 표준 ElementTree만 사용한다.

한 섹션 XML을 한 번 순회해 문단 텍스트, 표(셀 텍스트와 서식), 그림 이름을 함께 꺼낸다.
"""

import re
import zipfile
from xml.etree import ElementTree as ET

HP = "{http://www.hancom.co.kr/hwpml/2011/paragraph}"
HH = "{http://www.hancom.co.kr/hwpml/2011/head}"
HC = "{http://www.hancom.co.kr/hwpml/2011/core}"

SECTION_NAME = re.compile(r"Contents/section(\d+)\.xml")
IMAGE_NAME = re.compile(r"원본 그림의 이름:\s*([^\n]+)")
NO_FILL = (None, "", "none", "#FFFFFF", "#ffffff")


def is_hwpx(path) -> bool:
    with zipfile.ZipFile(path) as archive:
        return any(SECTION_NAME.fullmatch(n) for n in archive.namelist())


def section_names(archive: zipfile.ZipFile) -> list[str]:
    names = [n for n in archive.namelist() if SECTION_NAME.fullmatch(n)]
    return sorted(names, key=lambda n: int(SECTION_NAME.fullmatch(n).group(1)))


def para_text(paragraph: ET.Element) -> str:
    """변경추적 반영본 텍스트. deleteBegin~deleteEnd 구간은 버린다.

    표 셀·캡션 안에 중첩된 <hp:p>는 내려가지 않는다. 그 문단들은 섹션 순회에서
    따로 나오므로, 내려가면 바깥 문단에 셀 텍스트가 전부 합쳐진다.
    <hp:t> 안 인라인 요소 뒤의 tail 텍스트는 포함한다.
    """
    out, state = [], None

    def walk(element: ET.Element) -> None:
        nonlocal state
        for child in element:
            tag = child.tag.replace(HP, "")
            if tag == "p":
                continue
            if tag == "deleteBegin":
                state = "del"
            elif tag in ("deleteEnd", "insertEnd"):
                state = None
            elif tag == "insertBegin":
                state = "ins"
            if state != "del" and tag == "t":
                out.append(child.text or "")
            walk(child)
            if state != "del" and child.tail and element.tag == HP + "t":
                out.append(child.tail)

    walk(paragraph)
    return "".join(out)


def header_styles(archive: zipfile.ZipFile) -> tuple[dict, dict]:
    """header.xml → (charPr id→{bold, font}, borderFill id→면색 또는 None)."""
    root = ET.fromstring(archive.read("Contents/header.xml"))
    fonts = {}
    for face in root.iter(HH + "fontface"):
        if face.get("lang") == "HANGUL":
            for font in face.findall(HH + "font"):
                fonts[font.get("id")] = font.get("face")
    char_props = {}
    for prop in root.iter(HH + "charPr"):
        ref = prop.find(HH + "fontRef")
        char_props[prop.get("id")] = {
            "bold": prop.find(HH + "bold") is not None,
            "font": fonts.get(ref.get("hangul")) if ref is not None else None,
        }
    fills = {}
    for border_fill in root.iter(HH + "borderFill"):
        brush = border_fill.find(".//" + HC + "winBrush")
        color = brush.get("faceColor") if brush is not None else None
        fills[border_fill.get("id")] = None if color in NO_FILL else color.upper()
    return char_props, fills


def _cell_runs(cell: ET.Element, own_paragraphs: list[ET.Element], char_props: dict) -> list[dict]:
    runs = []
    for paragraph in own_paragraphs:
        for run in paragraph.iter(HP + "run"):
            text = re.sub(r"\s+", " ", "".join(t.text or "" for t in run.iter(HP + "t"))).strip()
            if text:
                runs.append(
                    {
                        "text": text,
                        **char_props.get(run.get("charPrIDRef"), {"bold": False, "font": None}),
                    }
                )
    return runs


def parse_section(root: ET.Element, section_index: int, styles: tuple[dict, dict] | None):
    """(문단 텍스트 목록, 표 목록, 그림 목록)을 돌려준다.

    표: {'section', 'paragraph', 'rows', 'cols', 'cells': {(r,c): 텍스트},
         'styles': {(r,c): {'fill', 'runs': [{'text','font','bold'}]}} (styles 없으면 None)}
    그림: (문단 인덱스, 원본 그림 이름)
    """
    parents = {child: parent for parent in root.iter() for child in parent}
    paragraphs = list(root.iter(HP + "p"))
    positions = {p: i for i, p in enumerate(paragraphs)}

    def owner_paragraph(element: ET.Element) -> ET.Element | None:
        current = parents.get(element)
        while current is not None and current.tag != HP + "p":
            current = parents.get(current)
        return current

    def owner_cell(paragraph: ET.Element) -> ET.Element | None:
        current = parents.get(paragraph)
        while current is not None and current.tag != HP + "tc":
            current = parents.get(current)
        return current

    images = []
    for comment in root.iter(HP + "shapeComment"):
        match = IMAGE_NAME.search(comment.text or "")
        paragraph = owner_paragraph(comment)
        if match and paragraph in positions:
            images.append((positions[paragraph], match.group(1).strip()))

    tables = []
    for table in root.iter(HP + "tbl"):
        paragraph = owner_paragraph(table)
        if paragraph not in positions:
            continue
        cells, cell_styles = {}, {}
        for row in table.findall(HP + "tr"):
            for cell in row.findall(HP + "tc"):
                address = cell.find(HP + "cellAddr")
                if address is None:
                    raise ValueError("HWPX 표 셀에 cellAddr가 없습니다")
                key = (int(address.get("rowAddr")), int(address.get("colAddr")))
                # 중첩 표는 자체 표로 처리하므로 바깥 셀에 합치지 않는다.
                own = [p for p in cell.iter(HP + "p") if owner_cell(p) is cell]
                cells[key] = re.sub(r"\s+", " ", " ".join(para_text(p) for p in own)).strip()
                if styles is not None:
                    char_props, fills = styles
                    cell_styles[key] = {
                        "fill": fills.get(cell.get("borderFillIDRef")),
                        "runs": _cell_runs(cell, own, char_props),
                    }
        tables.append(
            {
                "section": section_index,
                "paragraph": positions[paragraph],
                "rows": int(table.get("rowCnt") or 0),
                "cols": int(table.get("colCnt") or 0),
                "cells": cells,
                "styles": cell_styles if styles is not None else None,
            }
        )
    return [para_text(p) for p in paragraphs], tables, images


def load_hwpx(path):
    """(섹션별 문단, 표, 그림) — 그림은 (섹션, 문단, 이름)."""
    sections, tables, images = [], [], []
    with zipfile.ZipFile(path) as archive:
        try:
            styles = header_styles(archive)
        except KeyError:
            styles = None
        for index, name in enumerate(section_names(archive)):
            root = ET.fromstring(archive.read(name))
            paragraphs, section_tables, section_images = parse_section(root, index, styles)
            sections.append(paragraphs)
            tables.extend(section_tables)
            images.extend((index, paragraph, image) for paragraph, image in section_images)
    return sections, tables, images
''')


In [ ]:
# 모듈: ncs_core/inputs.py
write_module('ncs_core/inputs.py', r'''"""입력 파일의 역할 판별과 검증. Colab UI와 독립적이다."""

import hashlib
from pathlib import Path

from .documents import Document, load_document

ROLE_LABELS = {"manuscript": "원고", "plan": "개발계획서"}
MANUSCRIPT_MARKERS = ("차 례", "차례", "학습모듈의 개요", "NCS-학습모듈의 위치")


def detect_role(sections: list[list[str]], head_lines: int = 60) -> str:
    """첫 섹션 머리 문단으로 역할을 판별한다: 'plan' / 'manuscript' / 'unknown'."""
    head = " ".join(sections[0][:head_lines]) if sections else ""
    if "개발계획서" in head:
        return "plan"
    if any(marker in head for marker in MANUSCRIPT_MARKERS):
        return "manuscript"
    return "unknown"


def file_digest(path) -> bytes:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.digest()


def validate_distinct(paths: dict[str, str | Path]) -> None:
    """역할별 경로가 존재하고 비어 있지 않으며 서로 다른 내용인지 확인한다."""
    for role, path in paths.items():
        label = ROLE_LABELS.get(role, role)
        if not path:
            raise ValueError(f"{label} 파일 경로를 지정하세요")
        path = Path(path)
        if not path.is_file():
            raise ValueError(f"{label} 파일을 찾을 수 없습니다: {path}")
        if path.stat().st_size == 0:
            raise ValueError(f"{label} 파일이 비어 있습니다: {path}")
    digests = {role: file_digest(path) for role, path in paths.items()}
    if len(set(digests.values())) != len(digests):
        raise ValueError("입력 파일들의 내용이 동일합니다. 실제 원고와 개발계획서를 각각 선택하세요")


def select_documents(
    paths,
    roles: dict | None = None,
    *,
    required: tuple[str, ...] = ("manuscript",),
) -> dict[str, Document]:
    """경로 목록을 읽어 {'manuscript': Document, 'plan': Document}로 나눈다.

    roles={경로: 'plan'|'manuscript'}로 자동 판별을 덮어쓴다.
    required에 있는 역할이 없거나, 한 역할에 파일이 둘 이상이면 ValueError.
    """
    overrides = {str(Path(p).resolve()): r for p, r in (roles or {}).items()}
    if set(overrides.values()) - set(ROLE_LABELS):
        raise ValueError("roles 값은 'manuscript' 또는 'plan'이어야 합니다")
    documents: dict[str, Document] = {}
    seen: set[str] = set()
    for path in paths:
        key = str(Path(path).resolve())
        if key in seen:
            raise ValueError(f"같은 입력 파일이 중복 지정되었습니다: {path}")
        seen.add(key)
        document = load_document(path)
        role = overrides.get(key, detect_role(document.sections))
        if role not in ROLE_LABELS:
            raise ValueError(f"파일 역할을 판단하지 못했습니다: {path}. roles로 역할을 지정하세요")
        if role in documents:
            raise ValueError(f"{ROLE_LABELS[role]} 파일이 둘 이상입니다. 역할마다 한 개씩 선택하세요")
        documents[role] = document
    unused = overrides.keys() - seen
    if unused:
        raise ValueError(f"roles에 입력 목록에 없는 경로가 있습니다: {sorted(unused)}")
    for role in required:
        if role not in documents:
            raise ValueError(f"{ROLE_LABELS[role]} 파일이 필요합니다")
    if len(documents) > 1:
        validate_distinct({role: doc.path for role, doc in documents.items()})
    return documents
''')


In [ ]:
# 모듈: ncs_core/models.py
write_module('ncs_core/models.py', r'''"""파이프라인 경계의 공통 데이터 계약. 체크리스트 행의 한국어 JSON 키는 기존과 같다."""

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal, TypedDict

ChecklistItem = TypedDict(
    "ChecklistItem",
    {
        "item_id": str,
        "region_path": str,
        "원고": Any,
        "체크": dict[str, Literal["O", "X", "△", "-"]],
        "판정": str,
        "비고": dict[str, str] | list[str],
        "수정 제안": Any,
    },
)


@dataclass
class ReviewResult:
    """한 번의 검사 결과. 저장 형식과 무관하다."""

    module_code: str
    checklist: list[ChecklistItem]
    findings: list[dict]
    diagnostics: dict[str, Any] = field(default_factory=dict)
    context: Any = None


@dataclass(frozen=True)
class SavedReport:
    directory: Path
    zip_path: Path
    summary: dict[str, Any]
''')


In [ ]:
# 모듈: ncs_core/module_code.py
write_module('ncs_core/module_code.py', r'''"""모듈 코드 검증과 파일명 추출. input()을 호출하지 않는다."""

import re
from pathlib import Path

CODE_PATTERN = r"LM[0-9]{10}_[0-9]+v[0-9]+"
CODE_IN_NAME = re.compile(r"(?<![A-Za-z0-9])" + CODE_PATTERN + r"(?![A-Za-z0-9])")


def validate_module_code(code: str) -> str:
    code = (code or "").strip()
    if not re.fullmatch(CODE_PATTERN, code):
        raise ValueError("모듈 코드는 LM + 숫자 10자리 + _ + 숫자 + v + 숫자 형식이어야 합니다")
    if int(code[8:10]) < 1 or int(code[10:12]) < 1:
        raise ValueError("세분류 번호와 능력단위 번호는 01 이상이어야 합니다")
    return code


def resolve_module_code(manuscript_path, override: str | None = "") -> str:
    """override가 있으면 검증해 쓰고, 없으면 원고 파일명에서 하나뿐인 코드를 찾는다."""
    if override and override.strip():
        return validate_module_code(override)
    candidates = list(dict.fromkeys(CODE_IN_NAME.findall(Path(manuscript_path).name)))
    if len(candidates) != 1:
        raise ValueError("원고 파일명의 모듈 코드가 없거나 여러 개입니다. module_code를 지정하세요")
    return validate_module_code(candidates[0])
''')


In [ ]:
# 모듈: ncs_core/reporting.py
write_module('ncs_core/reporting.py', r'''"""검사 결과를 항목별 JSON과 ZIP으로 저장한다. 파일 이름 규칙은 두 패키지가 같다.

<out_dir>/<코드>_<실행별 폴더>/
    <코드>_<항목 꼬리말>.json   항목별 체크리스트
    <코드>_<이름>_00_요약.json  검수결과(적합/부적합/판단불가) 종합과 항목별 집계
    <코드>_<이름>_지적사항.json  X·△만
    <코드>_<이름>_전체.json     모든 체크리스트 행
    <코드>_<이름>_진단.json     입력·읽기 상태
    <코드>_<이름>_검수결과.zip
"""

import json
import tempfile
import zipfile
from collections.abc import Sequence
from pathlib import Path

from .models import ReviewResult, SavedReport
from .module_code import validate_module_code
from .text import FAIL, PASS, UNDECIDED, VERDICTS


def overall_verdict(verdicts) -> str:
    """행 판정들의 종합: 부적합이 하나라도 있으면 부적합, 없고 판단불가가 있으면 판단불가, 나머지는 적합."""
    verdicts = list(verdicts)
    if FAIL in verdicts:
        return FAIL
    if UNDECIDED in verdicts:
        return UNDECIDED
    return PASS


def save_report(
    result: ReviewResult,
    out_dir,
    *,
    name: str,
    item_files: Sequence[tuple[str, str]],
    other_tail: str,
    per_run_dir: bool = True,
) -> SavedReport:
    """name: 결과 이름('차례'/'위치'). item_files: (item_id 접두, 파일 꼬리말).

    per_run_dir=True이면 실행마다 새 하위 폴더를 만들어 이전 결과를 덮어쓰지 않는다.
    False이면 out_dir에 바로 쓰고 같은 코드의 이전 결과를 갱신한다.
    item_files에 없는 item_id는 other_tail 파일로 모인다.
    """
    code = validate_module_code(result.module_code)
    root = Path(out_dir).resolve()
    root.mkdir(parents=True, exist_ok=True)
    directory = Path(tempfile.mkdtemp(prefix=f"{code}_", dir=root)) if per_run_dir else root

    buckets: dict[str, list] = {tail: [] for _, tail in item_files}
    by_length = sorted(item_files, key=lambda pair: -len(pair[0]))
    for row in result.checklist:
        tail = next((t for prefix, t in by_length if row["item_id"].startswith(prefix)), other_tail)
        buckets.setdefault(tail, []).append(row)

    paths: list[Path] = []
    summary: dict = {}

    def write(tail: str, value) -> Path:
        path = directory / f"{code}_{tail}.json"
        path.write_text(json.dumps(value, ensure_ascii=False, indent=1), encoding="utf-8")
        paths.append(path)
        return path

    for tail, rows in buckets.items():
        path = write(tail, rows)
        verdicts = [row["판정"] for row in rows]
        summary[tail] = {"검수결과": overall_verdict(verdicts), "행 수": {v: verdicts.count(v) for v in VERDICTS}, "파일": path.name}
    summary = {"검수결과": overall_verdict(row["판정"] for row in result.checklist), "항목": summary}
    write(f"{name}_00_요약", summary)
    write(f"{name}_지적사항", result.findings)
    write(f"{name}_전체", result.checklist)
    write(f"{name}_진단", result.diagnostics)

    zip_path = directory / f"{code}_{name}_검수결과.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in paths:
            archive.write(path, path.name)
    return SavedReport(directory, zip_path, summary)
''')


In [ ]:
# 모듈: ncs_core/rules.py
write_module('ncs_core/rules.py', r'''"""규칙 등록과 실행. 규칙 함수는 컨텍스트 하나를 받아 체크리스트 행 목록을 돌려준다."""

from collections.abc import Callable, Sequence
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Rule:
    """key: 규칙 식별자(중복 불가). item_prefix: 반환 행의 item_id가 시작해야 하는 문자열."""

    key: str
    item_prefix: str
    run: Callable[[Any], list[dict]]


def run_rules(context: Any, rules: Sequence[Rule]) -> list[dict]:
    keys = [rule.key for rule in rules]
    if len(keys) != len(set(keys)):
        raise ValueError("중복된 규칙 key가 있습니다")
    rows: list[dict] = []
    for rule in rules:
        result = rule.run(context)
        for row in result:
            if not row["item_id"].startswith(rule.item_prefix):
                raise ValueError(f"{rule.key} 규칙이 다른 item_id를 반환했습니다: {row['item_id']}")
        rows.extend(result)
    return rows
''')


In [ ]:
# 모듈: ncs_core/tables.py
write_module('ncs_core/tables.py', r'''"""Document.tables(문서 순서의 표 목록)를 다루는 작은 도우미."""

from .documents import Table


def column(table: Table, col: int) -> list[str]:
    """해당 열의 비어 있지 않은 셀 텍스트를 행 순서대로."""
    return [v for (r, c), v in sorted(table["cells"].items()) if c == col and v]


def column_cells(table: Table, col: int, header: str | None = None) -> list[tuple[int, str]]:
    """해당 열의 비어 있지 않은 셀을 (행, 텍스트)로, header 텍스트는 제외."""
    return [
        (r, v)
        for (r, c), v in sorted(table["cells"].items())
        if c == col and v and v != header
    ]


def header_columns(table: Table, names: tuple[str, ...]) -> dict[str, int]:
    """names에 있는 머리글 셀 텍스트 → 열 번호 (먼저 나온 것 우선)."""
    found: dict[str, int] = {}
    for (r, c), value in sorted(table["cells"].items()):
        if value in names and value not in found:
            found[value] = c
    return found


def find_table(tables: list[Table], must_contain: str, after: Table | None = None) -> Table | None:
    """must_contain을 포함한 셀이 있는 첫 표 (after 표 다음부터)."""
    start = tables.index(after) + 1 if after is not None else 0
    for table in tables[start:]:
        if any(must_contain in v for v in table["cells"].values()):
            return table
    return None
''')


In [ ]:
# 모듈: ncs_core/text.py
write_module('ncs_core/text.py', r'''"""문자 정규화·비교·판정·지적사항 변환. 기존 결과와 호환된다.

정규화에서 어떤 머리 번호를 지우는지는 패키지마다 다르므로
`make_normalizer(prefix)`로 각 패키지가 자기 규칙의 `norm`/`same`을 만든다.
"""

import re
from collections.abc import Callable, Iterable

PUA = re.compile(r"[-]")
O, X, TRI, NA = "O", "X", "△", "-"
# 검수 확인서의 검수결과 표현. 한 항목의 판정은 이 셋 중 하나다.
PASS, FAIL, UNDECIDED = "적합", "부적합", "판단불가"
VERDICTS = (PASS, FAIL, UNDECIDED)

Normalizer = Callable[[str | None], str]


def make_normalizer(prefix: str | None = None) -> tuple[Normalizer, Callable[..., bool]]:
    """(norm, same)을 만든다.

    norm: 기호 글꼴(PUA) 문자를 가운뎃점으로 바꾸고, 양끝 공백과 머리 번호(prefix)를
    지운 뒤 연속 공백을 하나로 만든다.
    same: 두 문자열의 norm 결과가 같은가.
    """
    head = re.compile(prefix) if prefix else None

    def norm(text: str | None) -> str:
        value = PUA.sub("·", text or "").strip()
        if head is not None:
            value = head.sub("", value)
        return re.sub(r"\s+", " ", value)

    def same(expected: str | None, actual: str | None) -> bool:
        return norm(expected) == norm(actual)

    return norm, same


def visible(text: str | None) -> str:
    """PUA 문자를 ⟨U+XXXX⟩로 드러내 결과에 적는다."""
    return PUA.sub(lambda m: f"⟨U+{ord(m.group()):04X}⟩", text or "")


def verdict(checks: dict[str, str]) -> str:
    """X가 있으면 부적합, X 없이 △가 있으면 판단불가, 나머지는 적합. '-'는 무시."""
    values = [v for v in checks.values() if v != NA]
    if X in values:
        return FAIL
    if TRI in values:
        return UNDECIDED
    return PASS


def flatten(checklist: Iterable[dict]) -> list[dict]:
    """체크 항목 중 X·△만 지적사항 행으로 편다 (기존 키 유지)."""
    findings = []
    for row in checklist:
        for name, value in row["체크"].items():
            if value not in (X, TRI):
                continue
            notes = row["비고"]
            reason = notes.get(name, "") if isinstance(notes, dict) else "; ".join(notes)
            findings.append(
                {
                    "item_id": row["item_id"],
                    "check": name,
                    "region_path": row["region_path"],
                    "verdict": FAIL if value == X else UNDECIDED,
                    "evidence_quote": row["원고"],
                    "reason": reason,
                    "suggestion": row["수정 제안"],
                }
            )
    return findings
''')


In [ ]:
# 모듈: ncs_toc/__init__.py
write_module('ncs_toc/__init__.py', r'''"""F02 차례 ①·③·⑤·⑥ 검수. 읽기·판정·저장은 ncs_core를 쓴다."""

from ncs_core import ReviewResult, SavedReport

from .config import CheckConfig
from .pipeline import evaluate_review, prepare_review, run_review
from .registry import DEFAULT_RULES
from .reporting import save_results

__all__ = [
    "CheckConfig",
    "DEFAULT_RULES",
    "ReviewResult",
    "SavedReport",
    "prepare_review",
    "evaluate_review",
    "run_review",
    "save_results",
]
__version__ = "2.1.0"
''')


In [ ]:
# 모듈: ncs_toc/__main__.py
write_module('ncs_toc/__main__.py', r'''"""python -m ncs_toc 원고.hwpx [개발계획서.hwpx] [--module-code LM...] [--out 결과]"""

import argparse
import json

from . import CheckConfig, run_review, save_results


def main() -> None:
    parser = argparse.ArgumentParser(description="F02 차례 ①·③·⑤·⑥ 검수")
    parser.add_argument("paths", nargs="+", help="원고와 (선택) 개발계획서. 순서 무관, 내용으로 역할 판별")
    parser.add_argument("--manuscript", help="자동 역할 판별 대신 원고로 지정할 입력 경로")
    parser.add_argument("--plan", help="자동 역할 판별 대신 개발계획서로 지정할 입력 경로")
    parser.add_argument("--module-code", default="", help="비우면 원고 파일명에서 추출")
    parser.add_argument("--front-section", type=int, default=0)
    parser.add_argument("--body-section", type=int, default=1)
    parser.add_argument("--out", default="결과")
    args = parser.parse_args()
    roles = {}
    if args.manuscript:
        roles[args.manuscript] = "manuscript"
    if args.plan:
        roles[args.plan] = "plan"
    try:
        config = CheckConfig(front_section=args.front_section, body_section=args.body_section)
        result = run_review(args.paths, roles=roles, module_code=args.module_code, config=config)
        saved = save_results(result, args.out)
    except (OSError, ValueError) as error:
        parser.exit(2, f"검수 실행 실패: {error}\n")
    print(json.dumps({"요약": saved.summary, "진단": result.diagnostics}, ensure_ascii=False, indent=2))
    print(f"지적사항 {len(result.findings)}건 · {saved.zip_path}")


if __name__ == "__main__":
    main()
''')


In [ ]:
# 모듈: ncs_toc/compare.py
write_module('ncs_toc/compare.py', r'''"""목록 비교 도우미. 규칙 모듈이 함께 쓴다. 비교 정책은 v1.1을 유지한다."""

import difflib
from collections.abc import Iterable, Sequence

from ncs_core import O, PUA, X, visible

from .text import norm


def diff_list(label: str, expected: Sequence[str], actual: Sequence[str]) -> tuple[str, list[str]]:
    """difflib로 누락/추가/변경/순서를 구분한다 → (체크값, 비고 목록).

    중간에 하나가 빠져도 뒤가 연쇄로 '다름'이 되지 않는다.
    """
    exp, act = [norm(x) for x in expected], [norm(x) for x in actual]
    notes: list[str] = []
    if len(expected) != len(actual):
        notes.append(f"{label} 개수: 이 자리 {len(actual)} / 기준 {len(expected)}")
    if exp != act and sorted(exp) == sorted(act):
        notes.append(f"{label} 순서 다름: 이 자리 {[visible(x) for x in actual]} / 기준 {list(expected)}")
        return X, notes
    matcher = difflib.SequenceMatcher(None, exp, act, autojunk=False)
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == "equal":
            continue
        if op == "replace" and i2 - i1 == j2 - j1:
            for k in range(i2 - i1):
                notes.append(
                    f'{label} {i1 + k + 1}번째 다름: "{visible(actual[j1 + k])}" / 기준 "{expected[i1 + k]}"'
                )
        else:
            for k in range(i1, i2):
                notes.append(f'{label} {k + 1}번째 누락: "{expected[k]}"')
            for k in range(j1, j2):
                notes.append(f'{label} 이 자리 {k + 1}번째가 기준에 없음: "{visible(actual[k])}"')
    return (O if not notes else X), notes


def pua_note(texts: Iterable[str], already: Iterable[str] = ()) -> tuple[str, list[str]]:
    """already: 차례(①)에서 이미 보고된 문자열(PUA→· 치환형). 같은 원인이면 X 대신 비고만."""
    already = set(already)
    bad = [t for t in texts if PUA.search(t)]
    new = [t for t in bad if PUA.sub("·", t) not in already]
    notes = [f"기호 글꼴 문자: {visible(t)}" for t in new]
    if bad and not new:
        notes.append("기호 글꼴 문자가 있으나 차례와 같은 원인 (①에서 보고됨)")
    return (X if new else O), notes
''')


In [ ]:
# 모듈: ncs_toc/config.py
write_module('ncs_toc/config.py', r'''"""차례 검수의 조정 가능한 양식 가정. 정규식 자체는 patterns.py에서 관리한다."""
from dataclasses import dataclass


@dataclass(frozen=True)
class CheckConfig:
    front_section: int = 0
    body_section: int = 1
    body_heading_max_length: int = 60
    eval_candidate_max_length: int = 30
    tail_min_paragraphs: int = 3
    reference_min_lines: int = 3
    ambiguous_tail_min_paragraphs: int = 10

    def __post_init__(self):
        if self.front_section < 0 or self.body_section < 0:
            raise ValueError("섹션 번호는 0 이상이어야 합니다")
        if self.front_section == self.body_section:
            raise ValueError("현재 파서는 차례·개요와 본문이 별도 섹션인 양식을 지원합니다")
        for name in ("body_heading_max_length", "eval_candidate_max_length",
                     "tail_min_paragraphs", "reference_min_lines",
                     "ambiguous_tail_min_paragraphs"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name}은 1 이상이어야 합니다")


DEFAULT_CONFIG = CheckConfig()
''')


In [ ]:
# 모듈: ncs_toc/evaluation.py
write_module('ncs_toc/evaluation.py', r'''"""본문 표 가운데 평가 준거·평가 방법 표를 찾아 '학습 내용' 열 값을 꺼낸다."""

from ncs_core import Table

HEADER_KEYS = ("학습내용", "학습목표", "평가항목")


def evaluation_tables(tables: list[Table]) -> list[tuple[int, str, list[str]]]:
    """[(표가 있는 문단 인덱스, '평가 준거'|'평가 방법', ['학습 내용' 열 값 …])].

    '학습 내용' 머리글이 있고 '학습 목표'(평가 준거) 또는 '평가 항목'(평가 방법) 머리글이
    함께 있는 표만 평가 표로 본다.
    """
    out = []
    for table in tables:
        headers: dict[str, tuple[int, int]] = {}
        for (row, col), value in sorted(table["cells"].items()):
            key = next((k for k in HEADER_KEYS if value.replace(" ", "").startswith(k)), None)
            if key and key not in headers:
                headers[key] = (row, col)
        if "학습내용" not in headers or not ({"학습목표", "평가항목"} & headers.keys()):
            continue
        kind = "평가 준거" if "학습목표" in headers else "평가 방법"
        header_row, content_col = headers["학습내용"]
        values = [
            v for (r, c), v in sorted(table["cells"].items()) if c == content_col and r > header_row and v
        ]
        out.append((table["paragraph"], kind, values))
    return out
''')


In [ ]:
# 모듈: ncs_toc/models.py
write_module('ncs_toc/models.py', r'''"""파서와 규칙 사이의 경계. 결과 모델(ReviewResult 등)은 ncs_core.models를 쓴다."""

from dataclasses import dataclass
from typing import Any

from ncs_core import Document

from .config import CheckConfig


@dataclass(frozen=True)
class ReviewContext:
    """읽은 문서와 추출 결과. 규칙은 이 객체 하나를 받는다."""

    manuscript: Document
    plan: Document | None
    module_code: str
    config: CheckConfig
    toc: dict[str, Any]
    overview: dict[str, Any] | None
    plan_toc: dict[str, Any] | None
    body: dict[str, Any]
    presence: dict[str, Any]
    eval_tables: list[tuple[int, str, list[str]]]
''')


In [ ]:
# 모듈: ncs_toc/parsers.py
write_module('ncs_toc/parsers.py', r'''"""문단에서 차례, 개요, 본문 구조, 개발계획서 명칭을 추출한다.

sections는 [차례·개요 문단, 본문 문단]의 두 섹션 목록이다(pipeline이 실제 섹션을 연결한다).
"""

import re

from .config import DEFAULT_CONFIG, CheckConfig
from .patterns import (
    BODY_SUB,
    EVAL_ITEM_END,
    EVAL_LABELS,
    HEAD_EVAL,
    HEAD_TEACH,
    NAV_HEAD,
    OV_LESSON,
    OV_SUB,
    TOC_LESSON,
    TOC_LINE,
    TOC_SUB,
    TOC_TAIL,
)

TAIL_NAMES = {"참고자료": "참고 자료", "활용서식": "활용 서식", "부록": "부록"}


def parse_toc(sections) -> dict | None:
    """차례 블록 → {'학습': [{'no','text'}], '내용': [{'no','sub','text','page'}],
    '줄': [{'lesson','kind','page'}], '꼬리': {참고 자료: 쪽, ...}, 'span': (s, e)}"""
    lines = [l for l in sections[0] if l] if sections else []
    start = next((i for i, l in enumerate(lines) if re.fullmatch(r"차\s*례", l)), None)
    if start is None:
        return None
    end = next(
        (
            i
            for i in range(start + 1, len(lines))
            if re.search(r"학습모듈의\s*개요$", lines[i]) and not re.search(r"\d+$", lines[i])
        ),
        len(lines),
    )
    toc = {"학습": [], "내용": [], "줄": [], "꼬리": {}, "span": (start, end)}
    for line in lines[start + 1 : end]:
        if (m := TOC_LESSON.match(line)) and int(m.group(1)) == len(toc["학습"]) + 1:
            toc["학습"].append({"no": int(m.group(1)), "text": m.group(2).strip()})
        elif m := TOC_SUB.match(line):
            toc["내용"].append(
                {"no": int(m.group(1)), "sub": int(m.group(2)), "text": m.group(3).strip(), "page": m.group(4)}
            )
        elif m := TOC_LINE.match(line):
            kind = "교수·학습 방법" if "교수" in m.group(1) else "평가"
            toc["줄"].append({"lesson": len(toc["학습"]), "kind": kind, "page": m.group(2)})
        elif m := TOC_TAIL.match(line):
            toc["꼬리"][TAIL_NAMES[re.sub(r"\s+", "", m.group(1))]] = m.group(2)
    return toc


def parse_overview(sections) -> dict | None:
    """학습모듈의 개요 > 내용체계 → {'학습': [{'no','text'}], '내용': [{'no','sub','text'}]}"""
    lines = [l for l in sections[0] if l] if sections else []
    start = next((i for i, l in enumerate(lines) if re.fullmatch(r"학습모듈의\s*내용\s*체계", l)), None)
    if start is None:
        return None
    end = next((i for i in range(start + 1, len(lines)) if re.fullmatch(r"핵심\s*용어", lines[i])), len(lines))
    overview = {"학습": [], "내용": []}
    for line in lines[start + 1 : end]:
        if m := OV_SUB.match(line):
            overview["내용"].append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": m.group(3).strip()})
        elif (m := OV_LESSON.match(line)) and int(m.group(1)) == len(overview["학습"]) + 1:
            overview["학습"].append({"no": int(m.group(1)), "text": m.group(2).strip()})
    return overview


def _navigator_runs(lines, n_lessons: int) -> list[list[dict]]:
    """내비게이터: '학습 1' 다음 줄이 이름, 이어서 '학습 2' …가 한 묶음(run)."""
    runs, k = [], 0
    while k < len(lines):
        head = NAV_HEAD.match(lines[k][1])
        if head and int(head.group(1)) == 1:
            run, no, j = [], 1, k
            while j + 1 < len(lines):
                nav = NAV_HEAD.match(lines[j][1])
                if not (nav and int(nav.group(1)) == no):
                    break
                run.append({"no": no, "text": lines[j + 1][1], "k": j})
                no += 1
                j += 2
            # 개수 차이는 ③ 내비게이터 항목의 비고로 보고한다.
            if len(run) >= (1 if n_lessons == 1 else 2):
                runs.append(run)
                k = j
                continue
        k += 1
    return runs


def _eval_candidates(lines, start: int, end: int, config: CheckConfig) -> list[str]:
    """평가 블록 안의 짧은 줄 = '학습 내용' 후보 (표를 못 읽을 때의 휴리스틱)."""
    out = []
    for _, line in lines[start:end]:
        if (
            len(line) <= config.eval_candidate_max_length
            and not re.match(r"^[-•·※\d(\[]", line)
            and line not in EVAL_LABELS
            and not NAV_HEAD.match(line)
            and not HEAD_EVAL.match(line)
            and not re.search(r"[.?!]$", line)
            and not EVAL_ITEM_END.search(line)
        ):
            out.append(line)
    return out


def parse_body(sections, n_lessons: int, cutoff: int | None = None, config: CheckConfig = DEFAULT_CONFIG) -> dict:
    """본문 → 내비게이터 묶음 / 학습 구간 / 필요지식 N-M. 제목 / 학습별 교수·평가 제목 위치와 평가 후보.

    cutoff: 참고 자료 시작 인덱스(raw) — 마지막 학습 구간을 여기서 끊는다.
    """
    raw = sections[1] if len(sections) > 1 else []
    lines = [(i, l) for i, l in enumerate(raw) if l]  # (원래 인덱스, 텍스트)
    nav_runs = _navigator_runs(lines, n_lessons)
    starts = [run[0]["k"] for run in nav_runs]
    last = next((k for k, (i, _) in enumerate(lines) if cutoff is not None and i >= cutoff), len(lines))
    spans = [(i + 1, starts[i], starts[i + 1] if i + 1 < len(starts) else last) for i in range(len(starts))]

    subs = []  # 필요지식 N-M. 제목 (같은 줄 또는 다음 줄)
    for k, (_, line) in enumerate(lines):
        if (m := BODY_SUB.match(line)) and len(line) < config.body_heading_max_length:
            text = m.group(3).strip() or (lines[k + 1][1] if k + 1 < len(lines) else "")
            if not re.search(r"\s\d+$", line):  # 차례 줄(끝 쪽수) 제외
                subs.append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": text, "k": k})

    lessons = []
    for no, a, b in spans:
        segment = lines[a:b]
        teach = [a + j for j, (_, l) in enumerate(segment) if HEAD_TEACH.match(l)]
        evals = [a + j for j, (_, l) in enumerate(segment) if HEAD_EVAL.match(l)]
        candidates = _eval_candidates(lines, evals[0], b, config) if evals else []
        own_subs = [s for s in subs if a <= s["k"] < b and s["no"] == no]
        raw_end = lines[b][0] if b < len(lines) else (lines[-1][0] + 1 if lines else 0)
        lessons.append(
            {
                "no": no,
                "span": (a, b),
                "raw_span": (lines[a][0], raw_end),
                "teach": teach,
                "eval": evals,
                "eval_cands": candidates,
                "subs": own_subs,
            }
        )
    return {"lines": lines, "nav_runs": nav_runs, "spans": spans, "subs": subs, "lessons": lessons}


def parse_plan_toc(sections, module_code: str) -> dict | None:
    """개발계획서 → {'학습명': [...], '요소명': [...], '내용명': [{'no','sub','text'}]} (블록을 못 찾으면 None)"""
    lines = [l for section in sections for l in section]
    code = module_code.replace("LM", "")
    start = next(
        (i for i, l in enumerate(lines) if l == code and i > 0 and lines[i - 1].startswith("능력단위 코드번호")),
        None,
    )
    if start is None:
        return None
    end = next((i for i in range(start, len(lines)) if lines[i].startswith("NCS (수정 금지)")), len(lines))
    block = lines[start:end]
    plan = {"학습명": [], "요소명": [], "내용명": []}
    for i, line in enumerate(block):
        if re.fullmatch(rf"{re.escape(code)}\.\d+", line) and i + 1 < len(block):
            plan["요소명"].append(block[i + 1])
        if m := OV_SUB.match(line):
            plan["내용명"].append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": m.group(3).strip()})
        elif re.fullmatch(r"\d+\.\s+\S.*", line):
            plan["학습명"].append(line)
    return plan
''')


In [ ]:
# 모듈: ncs_toc/patterns.py
write_module('ncs_toc/patterns.py', r'''"""양식별 제목·내용 인식 패턴. 판정 정책은 rules/에 둔다."""
import re

TOC_LESSON = re.compile(r'^학습\s*(\d+)\.\s*(.+?)(?:\s+\d+)?$')                 # 학습 1. 명 [쪽]
TOC_SUB    = re.compile(r'^(\d+)-(\d+)\.\s*(.+?)(?:\s*(\d+))?$')                 # 1-1. 명 [쪽]  ('• 평가52' 처럼 붙은 쪽수도)
TOC_LINE   = re.compile(r'^[•·]\s*(교수\s*\W{0,3}\s*학습\s*방법|평가)\s*(\d+)?$')
TOC_TAIL   = re.compile(r'^(참고\s*자료|활용\s*서식|부록)\s*(\d+)?$')
OV_LESSON  = re.compile(r'^(\d+)\.\s*(\S.*)$')                                   # 개요 내용체계: 1. 명
OV_SUB     = re.compile(r'^(\d+)-(\d+)\.\s*(\S.*)$')
NAV_HEAD   = re.compile(r'^학습\s*(\d+)$')
BODY_SUB   = re.compile(r'^(\d+)-(\d+)\.\s*(.*)$')
HEAD_TEACH = re.compile(r'^교수\s*\W{0,3}\s*학습\s*방법$')
HEAD_EVAL  = re.compile(r'^평\s*가$')
REF_LINE   = re.compile(r'^[•·]?\s*\S.{0,80}?\(\d{4}[a-z]?\)\s*[.,]')          # • 저자(연도).
EVAL_LABELS = {'평가 준거', '학습 내용', '학습 목표', '성취수준', '상', '중', '하', '평가 방법', '평가 항목', '피드백', '학습내용'}
EVAL_ITEM_END = re.compile(r'(여부|능력|수준|정도|가능)\s*$')     # 평가 항목 줄 종결어 (학습 내용명 후보에서 제외)
''')


In [ ]:
# 모듈: ncs_toc/pipeline.py
write_module('ncs_toc/pipeline.py', r'''"""입력 선택 → 추출 → 규칙 실행. 저장과 화면 출력은 호출자가 결정한다.

prepare_review: 문서를 읽고 ReviewContext를 만든다.
evaluate_review: 규칙을 실행해 ReviewResult를 만든다.
run_review: 둘을 이어 붙인 편의 함수.
"""

import re
from collections.abc import Sequence
from typing import Any

from ncs_core import Document, ReviewResult, Rule, resolve_module_code, run_rules, select_documents
from ncs_core.text import flatten

from .config import DEFAULT_CONFIG, CheckConfig
from .evaluation import evaluation_tables
from .models import ReviewContext
from .parsers import parse_body, parse_overview, parse_plan_toc, parse_toc
from .presence import section_presence
from .registry import ALL_ITEM_IDS, DEFAULT_RULES


def prepare_review(
    manuscript: Document,
    plan: Document | None = None,
    *,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
) -> ReviewContext:
    code = resolve_module_code(manuscript.path, module_code)
    if max(config.front_section, config.body_section) >= len(manuscript.sections):
        raise ValueError("지정한 차례/본문 섹션이 없습니다. 문서 구조와 CheckConfig를 확인하세요")
    # 파서의 논리적 section0(차례·개요)/section1(본문)에 양식의 실제 섹션을 연결한다.
    sections = [manuscript.sections[config.front_section], manuscript.sections[config.body_section]]
    toc = parse_toc(sections)
    if not toc or not toc["학습"]:
        raise ValueError("차례 또는 차례 학습명을 읽지 못했습니다. 양식과 섹션 설정을 확인하세요")
    overview = parse_overview(sections)
    plan_toc = parse_plan_toc(plan.sections, code) if plan else None
    images = [(p, name) for s, p, name in manuscript.images if s == config.body_section]
    presence = section_presence(sections, images, config)
    body = parse_body(sections, len(toc["학습"]), cutoff=presence["참고 자료"]["k"], config=config)
    eval_tables = evaluation_tables(manuscript.section_tables(config.body_section))
    return ReviewContext(manuscript, plan, code, config, toc, overview, plan_toc, body, presence, eval_tables)


def _actual_sections(value: Any, config: CheckConfig) -> Any:
    """결과의 'section0/section1' 표기를 실제 입력 섹션 번호로 바꾼다."""
    if isinstance(value, str):
        return re.sub(
            r"section([01])\b",
            lambda m: "section" + str(config.front_section if m[1] == "0" else config.body_section),
            value,
        )
    if isinstance(value, dict):
        return {k: _actual_sections(v, config) for k, v in value.items()}
    if isinstance(value, list):
        return [_actual_sections(v, config) for v in value]
    return value


def _diagnostics(context: ReviewContext, rules: Sequence[Rule]) -> dict:
    manuscript, plan, config = context.manuscript, context.plan, context.config
    warnings = []
    if not context.plan_toc:
        warnings.append("개발계획서가 없거나 해당 모듈 블록을 읽지 못했습니다")
    elif not context.plan_toc["학습명"] or not context.plan_toc["내용명"]:
        warnings.append("개발계획서 명칭을 일부만 읽었습니다. 항목별 비교 기준을 확인하세요")
    if not context.body["lessons"]:
        warnings.append("본문 학습 구간을 읽지 못했습니다. 평가 대조는 판단불가입니다")
    if not context.eval_tables:
        warnings.append("평가 표 셀 정보가 없어 문단 휴리스틱으로 비교합니다")
    if len(manuscript.sections) > 2:
        warnings.append("선택한 차례/개요 및 본문 섹션만 검사합니다. 다른 섹션에 이어진 본문은 검사하지 않습니다")
    implemented = {rule.item_prefix for rule in rules}
    return {
        "module_code": context.module_code,
        "inputs": {"manuscript": str(manuscript.path), "plan": str(plan.path) if plan else None},
        "format": manuscript.format,
        "sections": {"front": config.front_section, "body": config.body_section},
        "counts": {
            "lessons": len(context.toc["학습"]),
            "contents": len(context.toc["내용"]),
            "navigator_runs": len(context.body["nav_runs"]),
            "evaluation_tables": len(context.eval_tables),
        },
        "implemented": sorted(implemented),
        "not_implemented": [item for item in ALL_ITEM_IDS if item not in implemented],
        "warnings": warnings,
    }


def evaluate_review(context: ReviewContext, *, rules: Sequence[Rule] = DEFAULT_RULES) -> ReviewResult:
    """한 실행의 결과만 돌려준다. 같은 context를 다시 실행해도 이전 결과를 누적하지 않는다."""
    config = context.config
    checklist = run_rules(context, rules)
    for row in checklist:
        row["region_path"] = _actual_sections(row["region_path"], config)
        row["비고"] = _actual_sections(row["비고"], config)
        if row["item_id"] == "F02-차례-⑤":
            row["원고"] = _actual_sections(row["원고"], config)
    diagnostics = _diagnostics(context, rules)
    return ReviewResult(context.module_code, checklist, flatten(checklist), diagnostics, context)


def run_review(
    paths,
    *,
    roles: dict | None = None,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
    rules: Sequence[Rule] = DEFAULT_RULES,
) -> ReviewResult:
    """paths: 원고 1개 + 선택적 개발계획서 1개. roles={경로: 'manuscript'|'plan'}로 역할을 지정할 수 있다."""
    documents = select_documents(paths, roles, required=("manuscript",))
    manuscript, plan = documents["manuscript"], documents.get("plan")
    context = prepare_review(manuscript, plan, module_code=module_code, config=config)
    return evaluate_review(context, rules=rules)
''')


In [ ]:
# 모듈: ncs_toc/presence.py
write_module('ncs_toc/presence.py', r'''"""참고 자료·활용 서식·부록이 본문에 있는지의 근거를 찾는다."""

import re

from .config import DEFAULT_CONFIG, CheckConfig
from .patterns import REF_LINE

TAIL_PATTERNS = {"참고 자료": r"참\s*고\s*자\s*료", "활용 서식": r"활\s*용\s*서\s*식", "부록": r"부\s*록"}


def section_presence(sections, images, config: CheckConfig = DEFAULT_CONFIG) -> dict:
    """{이름: {'있음': bool|None, '근거': str, 'k': 위치}}.

    images: 본문 섹션의 (문단 인덱스, 원본 그림 이름). None은 '배너만 있고 내용은 없음'으로 사람 확인.
    """
    raw = sections[1] if len(sections) > 1 else []
    lines = [(i, l) for i, l in enumerate(raw) if l]
    out = {}
    for name, pattern in TAIL_PATTERNS.items():
        found = {"있음": False, "근거": "제목 텍스트·배너 이미지·내용 패턴 어느 것도 없음", "k": None}
        hit = next(((i, l) for i, l in lines if re.fullmatch(pattern, l)), None)
        if hit:  # 1) 제목 텍스트
            found = {"있음": True, "근거": f'제목 텍스트 "{hit[1]}" (section1 p{hit[0]})', "k": hit[0]}
        else:  # 2) 배너 이미지
            compact = pattern.replace(r"\s*", "")
            banner = next(((p, n) for p, n in images if re.search(compact, n.replace(" ", ""))), None)
            if banner:
                found = {"있음": True, "근거": f'배너 이미지 "{banner[1]}" (section1 p{banner[0]})', "k": banner[0]}
        out[name] = found

    # 제목·배너 뒤에 내용이 없으면(개발이력 전까지 문단이 적으면) '배너만' → △
    for name in ("활용 서식", "부록"):
        found = out[name]
        if not found["있음"]:
            continue
        others = [TAIL_PATTERNS[o] for o in TAIL_PATTERNS if o != name]
        stop = next(
            (
                i
                for i, l in lines
                if i > found["k"] and (re.search(r"개발\s*이력", l) or any(re.fullmatch(p, l) for p in others))
            ),
            len(raw),
        )
        body_n = len([l for i, l in lines if found["k"] < i < stop and not re.match(r"^※", l)])
        if body_n < config.tail_min_paragraphs:
            out[name] = {
                "있음": None,
                "근거": f'{found["근거"]} 뒤에 내용이 없음(문단 {body_n}개) — 양식 배너만 남은 것인지 확인',
                "k": found["k"],
            }

    # 3) 내용 패턴 — 참고 자료: 참고문헌 꼴 줄이 연속으로 충분히 있으면 존재로 본다
    if not out["참고 자료"]["있음"]:
        run, best = [], []
        for i, l in lines:
            if REF_LINE.match(l):
                run.append(i)
            else:
                if len(run) > len(best):
                    best = run
                run = []
        if len(run) > len(best):
            best = run
        if len(best) >= config.reference_min_lines:
            out["참고 자료"] = {
                "있음": True,
                "근거": f"참고문헌 꼴 줄 {len(best)}개 연속 (section1 p{best[0]}~p{best[-1]}) — 제목은 텍스트가 아님(이미지일 가능성)",
                "k": best[0],
                "end": best[-1],
            }

    # 활용 서식: 참고 자료 뒤 ~ 개발이력 앞에 내용이 있으면 '있을 가능성'
    if not out["활용 서식"]["있음"] and out["참고 자료"]["있음"]:
        start = out["참고 자료"].get("end", out["참고 자료"]["k"])
        stop = next((i for i, l in lines if i > start and re.search(r"개발\s*이력", l)), len(raw))
        middle = [l for i, l in lines if start < i < stop and not REF_LINE.match(l) and not re.match(r"^※", l)]
        if len(middle) >= config.ambiguous_tail_min_paragraphs:
            out["활용 서식"] = {
                "있음": None,
                "근거": (
                    f'제목은 없지만 참고 자료 뒤~개발이력 앞에 문단 {len(middle)}개 있음 (예: "{middle[0][:30]}") '
                    "— 활용 서식 또는 부록 내용으로 보임(제목이 이미지라 둘을 구분 못 함), 확인 필요"
                ),
                "k": start + 1,
            }
    return out
''')


In [ ]:
# 모듈: ncs_toc/registry.py
write_module('ncs_toc/registry.py', r'''"""규칙 추가 지점. 규칙은 ReviewContext 하나를 받아 기존 형식의 행 목록을 돌려준다."""

from ncs_core import Rule

from .rules.consistency import check_toc3
from .rules.names import check_toc1
from .rules.tail import check_toc5, check_toc6

DEFAULT_RULES = (
    Rule("names", "F02-차례-①", lambda c: check_toc1(c.plan_toc, c.toc)),
    Rule("consistency", "F02-차례-③", lambda c: check_toc3(c.plan_toc, c.toc, c.overview, c.body, c.eval_tables)),
    Rule("references", "F02-차례-⑤", lambda c: check_toc5(c.presence, c.toc)),
    Rule("appendix", "F02-차례-⑥", lambda c: check_toc6(c.presence, c.toc)),
)

# 차례 항목 ①~⑦ 중 규칙이 없는 것은 진단의 not_implemented로 보고한다.
ALL_ITEM_IDS = tuple("F02-차례-" + n for n in "①②③④⑤⑥⑦")
''')


In [ ]:
# 모듈: ncs_toc/reporting.py
write_module('ncs_toc/reporting.py', r'''"""차례 검수 결과 저장. 파일 이름 규칙은 ncs_core.reporting과 같다."""

from ncs_core import ReviewResult, SavedReport, save_report

ITEM_FILES = (
    ("F02-차례-①", "차례①_학습명"),
    ("F02-차례-③", "차례③_학습명내용명_5곳일치"),
    ("F02-차례-⑤", "차례⑤_참고자료"),
    ("F02-차례-⑥", "차례⑥_활용서식부록"),
)


def save_results(result: ReviewResult, out_dir="결과", *, per_run_dir: bool = True) -> SavedReport:
    return save_report(
        result, out_dir, name="차례", item_files=ITEM_FILES, other_tail="차례_기타", per_run_dir=per_run_dir
    )
''')


In [ ]:
# 모듈: ncs_toc/rules/__init__.py
write_module('ncs_toc/rules/__init__.py', r'''"""문서 읽기와 독립적인 차례 검수 규칙."""
''')


In [ ]:
# 모듈: ncs_toc/rules/consistency.py
write_module('ncs_toc/rules/consistency.py', r'''"""F02 차례 ③: 개발계획서를 기준으로 다섯 자리(차례·개요·내비게이터·필요지식·평가)의 명칭을 비교한다.

개발계획서를 못 읽으면 차례를 기준으로 비교하되 △로 둔다.
"""

from ncs_core import O, PASS, PUA, TRI, UNDECIDED, verdict, visible, X

from ..compare import diff_list, pua_note
from ..text import norm, same

ITEM_ID = "F02-차례-③"


def _row(place, region_path, *, manuscript, reference, checks, notes, suggestion):
    result = verdict(checks)
    return {
        "item_id": ITEM_ID,
        "region_path": region_path,
        "자리": place,
        "원고": manuscript,
        "기준": reference,
        "체크": checks,
        "판정": result,
        "비고": notes,
        "수정 제안": None if result == PASS else suggestion,
    }


def _unreadable(place, region_path, check_name, note):
    return {
        "item_id": ITEM_ID,
        "region_path": region_path,
        "자리": place,
        "원고": None,
        "기준": None,
        "체크": {check_name: TRI},
        "판정": UNDECIDED,
        "비고": {check_name: note},
        "수정 제안": None,
    }


def check_toc3(plan, toc, overview, body, eval_tables=None) -> list[dict]:
    rows = []
    use_plan_names = bool(plan and plan["학습명"])
    use_plan_subs = bool(plan and plan["내용명"])
    ref_names = [norm(x) for x in plan["학습명"]] if use_plan_names else [L["text"] for L in toc["학습"]]
    ref_subs = plan["내용명"] if use_plan_subs else toc["내용"]
    ref_sub_texts = [s["text"] for s in ref_subs]
    toc_texts = [L["text"] for L in toc["학습"]] + [s["text"] for s in toc["내용"]]
    toc_pua = {PUA.sub("·", t) for t in toc_texts if PUA.search(t)}
    reference = {
        "학습명": ref_names,
        "학습 내용명": ref_sub_texts,
        "출처": "개발계획서" if use_plan_names and use_plan_subs else "차례" if not use_plan_names and not use_plan_subs else "혼합",
        "학습명 출처": "개발계획서" if use_plan_names else "차례",
        "학습 내용명 출처": "개발계획서" if use_plan_subs else "차례",
    }
    base_note = {}
    if not use_plan_names:
        base_note["기준"] = "개발계획서 학습명을 읽지 못해 차례를 기준으로 비교 (△)"
    if not use_plan_subs:
        base_note["기준(내용명)"] = "개발계획서 학습 내용명을 읽지 못해 차례를 기준으로 비교 (△)"

    def item(place, act_names, act_subs, region_path, extra_checks=None, extra_notes=None):
        checks, notes = {}, dict(base_note)
        if act_names is not None:
            value, found = diff_list("학습명", ref_names, act_names)
            checks["학습명 일치"] = value
            if found:
                notes["학습명 일치"] = " / ".join(found)
            if not use_plan_names and value == O and place != "차례":
                checks["학습명 일치"] = TRI
                notes["학습명 일치"] = "차례와 일치 (개발계획서 미확인)"
        if act_subs is not None:
            value, found = diff_list("학습 내용명", ref_sub_texts, act_subs)
            checks["학습 내용명 일치"] = value
            if found:
                notes["학습 내용명 일치"] = " / ".join(found)
            if not use_plan_subs and value == O and place != "차례":
                checks["학습 내용명 일치"] = TRI
                notes["학습 내용명 일치"] = "차례와 일치 (개발계획서 미확인)"
        value, found = pua_note((act_names or []) + (act_subs or []), toc_pua)
        checks["기호 글꼴 문자 없음"] = value
        if found:
            notes["기호 글꼴 문자 없음"] = " / ".join(found)
        if extra_checks:
            checks.update(extra_checks)
        if extra_notes:
            notes.update(extra_notes)
        rows.append(
            _row(
                place,
                region_path,
                manuscript={
                    "학습명": [visible(x) for x in act_names] if act_names is not None else None,
                    "학습 내용명": [visible(x) for x in act_subs] if act_subs is not None else None,
                },
                reference=reference,
                checks=checks,
                notes=notes,
                suggestion="기준 값으로 통일",
            )
        )

    # 1 차례 — 개발계획서 기준일 때만 의미 있음(차례 기준이면 자기 자신이므로 PUA만)
    if use_plan_names or use_plan_subs:
        item(
            "차례",
            [L["text"] for L in toc["학습"]] if use_plan_names else None,
            [s["text"] for s in toc["내용"]] if use_plan_subs else None,
            "section0 > 차례",
        )
    # 2 개요 내용체계
    if overview:
        item(
            "학습모듈의 개요 > 내용체계",
            [L["text"] for L in overview["학습"]],
            [s["text"] for s in overview["내용"]],
            "section0 > 학습모듈의 개요 > 내용체계 표",
        )
    else:
        rows.append(_unreadable("내용체계", "section0 > 학습모듈의 개요", "표 찾기", "'학습모듈의 내용체계' 표를 찾지 못함"))
    # 3 내비게이터 — 첫 묶음을 비교하고, 나머지 묶음의 차이는 추가 체크로
    if body["nav_runs"]:
        extra_checks, extra_notes = {}, {}
        for r_i, run in enumerate(body["nav_runs"][1:], 2):
            value, found = diff_list("학습명", ref_names, [x["text"] for x in run])
            if value == X:
                key = f"내비게이터 {r_i}번째(학습 {r_i}) 일치"
                extra_checks[key] = X
                extra_notes[key] = " / ".join(found)
        if len(body["nav_runs"]) != len(toc["학습"]):
            extra_checks["내비게이터 반복 횟수"] = TRI
            extra_notes["내비게이터 반복 횟수"] = (
                f"내비게이터 묶음 {len(body['nav_runs'])}개 / 차례 학습 {len(toc['학습'])}개 — 학습 구간이 어긋났을 수 있음"
            )
        item(
            "내비게이터",
            [x["text"] for x in body["nav_runs"][0]],
            None,
            f"section1 > 내비게이터 (학습마다 반복 {len(body['nav_runs'])}회)",
            extra_checks,
            extra_notes,
        )
    else:
        rows.append(_unreadable("내비게이터", "section1 > 내비게이터", "내비게이터 찾기", "'학습 1'…'학습 N' 제목 묶음을 찾지 못함"))
    # 4 필요지식 N-M. 제목 — 학습 구간 안의 것만, (no, sub) 순서로
    body_subs = sorted({(s["no"], s["sub"]): s["text"] for L in body["lessons"] for s in L["subs"]}.items())
    item("필요지식(본문) 학습 내용명", None, [t for _, t in body_subs], "section1 > 본문 N-M. 제목")
    # 5 평가 — 학습마다. 1순위: 표의 '학습 내용' 열(확정). 2순위: 블록 안 짧은 줄 휴리스틱(△)
    rows.append(_evaluation_row(body, ref_subs, ref_sub_texts, eval_tables, use_plan_subs, toc_pua, base_note))
    return rows


def _evaluation_row(body, ref_subs, ref_sub_texts, eval_tables, use_plan_subs, toc_pua, base_note):
    checks, notes, seen = {}, {}, []
    by_table = bool(eval_tables)
    for lesson in body["lessons"]:
        expected = [s["text"] for s in ref_subs if s["no"] == lesson["no"]]
        key = f"학습 {lesson['no']} 평가"
        if by_table:
            start, end = lesson["raw_span"]
            tables = [(kind, values) for paragraph, kind, values in eval_tables if start <= paragraph < end]
            if not tables:
                checks[key] = X
                notes[key] = "학습 구간 안에 평가 준거·평가 방법 표('학습 내용' 열)를 찾지 못함"
                continue
            problems = []
            for kind, values in tables:
                seen += values
                value, found = diff_list(f"{kind} 학습 내용", expected, values)
                if value == X:
                    problems += found
            checks[key] = O if not problems else X
            if problems:
                notes[key] = " / ".join(problems)
            kinds = {kind for kind, _ in tables}
            missing = [k for k in ("평가 준거", "평가 방법") if k not in kinds]
            if missing:
                checks[key] = TRI if checks[key] == O else checks[key]
                notes[key] = (notes.get(key, "") + f" / {', '.join(missing)} 표('학습 내용' 열)를 찾지 못함").strip(" /")
        else:
            if not lesson["eval"]:
                checks[key] = X
                notes[key] = "'평 가' 제목을 찾지 못함"
                continue
            candidates = lesson["eval_cands"]
            seen += candidates
            missing = [e for e in expected if not any(same(e, c) for c in candidates)]
            strange = sorted({c for c in candidates if not any(same(e, c) for e in expected)})
            if missing:
                checks[key] = X
                notes[key] = "평가 표에 없는 학습 내용명: " + ", ".join(f'"{m}"' for m in missing)
            elif strange:
                checks[key] = TRI
                notes[key] = "기준에 없는 짧은 줄(학습 내용명일 수 있음): " + ", ".join(f'"{visible(x)}"' for x in strange[:5])
            else:
                checks[key] = TRI
                notes[key] = "표를 못 읽어 문단 휴리스틱으로 확인 (hwpx 권장)"
    if not body["lessons"]:
        checks["평가 구간 찾기"] = TRI
        notes["평가 구간 찾기"] = "학습 구간을 읽지 못해 평가 표 대조를 수행하지 못함"
    if not use_plan_subs:
        checks["개발계획서 기준 확인"] = TRI
        notes["개발계획서 기준 확인"] = "개발계획서 학습 내용명을 읽지 못해 차례를 기준으로 비교"
    value, found = pua_note(seen, toc_pua)
    checks["기호 글꼴 문자 없음"] = value
    all_notes = dict(base_note)
    all_notes.update(notes)
    if found:
        all_notes["기호 글꼴 문자 없음"] = " / ".join(found)
    return _row(
        "평가",
        "section1 > 평가 (평가 준거·평가 방법 표)",
        manuscript={
            "평가 표 학습 내용" if by_table else "학습 내용명 후보": sorted(set(seen)),
            "읽기": "표 셀" if by_table else "문단 휴리스틱",
        },
        reference={"학습 내용명": ref_sub_texts, "출처": "개발계획서" if use_plan_subs else "차례"},
        checks=checks,
        notes=all_notes,
        suggestion="평가 준거·평가 방법 표의 학습 내용명을 기준 값으로 통일",
    )
''')


In [ ]:
# 모듈: ncs_toc/rules/names.py
write_module('ncs_toc/rules/names.py', r'''"""F02 차례 ①: 학습명 — '~하기' 종결, 개발계획서 학습명 일치, 능력단위요소명 준용, 기호 글꼴 문자."""

import re

from ncs_core import FAIL, NA, O, PASS, PUA, TRI, verdict, visible, X

from ..text import norm, same

ITEM_ID = "F02-차례-①"


def check_toc1(plan: dict | None, toc: dict) -> list[dict]:
    rows = []
    n_plan = len(plan["학습명"]) if plan else None
    if plan and n_plan != len(toc["학습"]):
        rows.append(
            {
                "item_id": ITEM_ID,
                "region_path": "section0 > 차례",
                "원고": f"학습 {len(toc['학습'])}개",
                "개발계획서": f"학습 {n_plan}개",
                "체크": {"학습 개수 일치": X},
                "판정": FAIL,
                "비고": {"학습 개수 일치": "학습 누락 또는 추가"},
                "수정 제안": None,
            }
        )
    for index, lesson in enumerate(toc["학습"]):
        text = lesson["text"]
        exp_name = plan["학습명"][index] if plan and index < n_plan else None
        exp_elem = plan["요소명"][index] if plan and index < len(plan["요소명"]) else None
        ok_name = same(exp_name, text) if exp_name else None
        ok_elem = same(exp_elem, text) if exp_elem else None
        ws_only = ok_elem is False and re.sub(r"\s", "", norm(exp_elem)) == re.sub(r"\s", "", norm(text))
        if ws_only:
            ok_elem = True
        checks = {
            "'~하기' 종결": O if re.search(r"하기$", text) else X,
            "개발계획서 학습명 일치": NA if ok_name is None else (O if ok_name else X),
            "능력단위요소명 준용": NA if ok_elem is None else (O if ok_elem else TRI),
            "기호 글꼴 문자 없음": X if PUA.search(text) else O,
        }
        notes = {}
        if checks["'~하기' 종결"] == X:
            notes["'~하기' 종결"] = "'~하기'로 종결되지 않음"
        if ok_name is False:
            notes["개발계획서 학습명 일치"] = f'개발계획서 학습명과 다름: "{exp_name}"'
        if ok_elem is False:
            notes["능력단위요소명 준용"] = f'능력단위요소명 "{exp_elem}" 과 글자가 다름 — 요소를 합치거나 이름을 바꿨는지 확인'
        if ws_only:
            notes["능력단위요소명 준용(참고)"] = f'능력단위요소명 "{exp_elem}" 과 공백만 다름 — 통과 처리'
        if PUA.search(text):
            notes["기호 글꼴 문자 없음"] = "가운뎃점 등이 기호 글꼴(사용자 정의 영역) 문자 — ·(U+00B7)로 교체"
        if not plan:
            notes["개발계획서"] = "개발계획서를 읽지 못해 학습명·요소명 대조는 생략"
        result = verdict(checks)
        rows.append(
            {
                "item_id": ITEM_ID,
                "region_path": f"section0 > 차례 > 학습 {lesson['no']}",
                "원고": visible(text),
                "개발계획서": norm(exp_name) if exp_name else None,
                "능력단위요소명": exp_elem,
                "체크": checks,
                "판정": result,
                "비고": notes,
                "수정 제안": norm(exp_name) if (result != PASS and exp_name) else None,
            }
        )
    return rows
''')


In [ ]:
# 모듈: ncs_toc/rules/tail.py
write_module('ncs_toc/rules/tail.py', r'''"""F02 차례 ⑤·⑥: 참고 자료, 활용 서식·부록의 차례↔본문 일치."""

from ncs_core import O, PASS, TRI, verdict, X

ITEM_ID_5 = "F02-차례-⑤"
ITEM_ID_6 = "F02-차례-⑥"


def check_toc5(presence: dict, toc: dict) -> list[dict]:
    found = presence["참고 자료"]
    in_toc = "참고 자료" in toc["꼬리"]
    checks = {
        "⑤ 참고 자료 본문 존재": O if found["있음"] else X,
        "차례에 참고 자료 줄": O if in_toc else X,
    }
    notes = {"근거": found["근거"]}
    if not in_toc:
        notes["차례에 참고 자료 줄"] = '차례에 "참고 자료" 줄이 없음'
    if not found["있음"]:
        notes["⑤ 참고 자료 본문 존재"] = "본문에서 참고 자료를 찾지 못함 (필수 작성)"
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID_5,
            "region_path": "section1 > 참고 자료",
            "원고": found["근거"],
            "차례": toc["꼬리"].get("참고 자료"),
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else "참고 자료를 작성하고 차례에 쪽수와 함께 제시",
        }
    ]


def check_toc6(presence: dict, toc: dict) -> list[dict]:
    rows = []
    for name in ("활용 서식", "부록"):
        found, in_toc = presence[name], name in toc["꼬리"]
        toc_page = toc["꼬리"][name] or "" if in_toc else ""
        if found["있음"] is None:
            value = TRI
            note = f'본문: {found["근거"]}' + (f" / 차례에는 {name} {toc_page}" if in_toc else " / 차례에는 없음")
        elif in_toc and found["있음"]:
            value, note = O, f'차례·본문 모두 있음 — {found["근거"]}'
        elif in_toc and not found["있음"]:
            value, note = X, f'차례에는 "{name} {toc_page}" 이 있는데 본문에 없음 — 미작성이면 차례에서 삭제'
        elif not in_toc and found["있음"]:
            value, note = X, f'본문에 {name} 이 있는데({found["근거"]}) 차례에 없음 — 차례에 추가'
        else:
            value, note = O, "차례·본문 모두 없음 (해당 없음)"
        label = f"⑥ {name} 차례↔본문 일치"
        checks = {label: value}
        if value == O:
            suggestion = None
        elif in_toc and found["있음"] is False:
            suggestion = "차례에서 삭제"
        elif found["있음"] and not in_toc:
            suggestion = "차례에 추가"
        else:
            suggestion = "본문 확인 후 차례와 맞출 것"
        rows.append(
            {
                "item_id": ITEM_ID_6,
                "region_path": f"section0 > 차례 > {name}",
                "원고": {"차례": toc["꼬리"].get(name) if in_toc else None, "본문": found["있음"]},
                "체크": checks,
                "판정": verdict(checks),
                "비고": {label: note} if value != O or "해당 없음" in note else {},
                "수정 제안": suggestion,
            }
        )
    return rows
''')


In [ ]:
# 모듈: ncs_toc/text.py
write_module('ncs_toc/text.py', r'''"""차례 검수의 문자 비교 규칙. 머리 번호 '학습 1. ' / '1. ' / '1-1. '을 지우고 비교한다."""

from ncs_core import make_normalizer

HEAD_NUMBER = r"^(학습\s*)?\d+(-\d+)?\.\s*"
norm, same = make_normalizer(HEAD_NUMBER)
''')


In [ ]:
# 재실행해도 이전 버전의 import 캐시를 사용하지 않습니다.
for name in list(sys.modules):
    if name.split(".")[0] in ("ncs_core", "ncs_toc"):
        del sys.modules[name]
if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))
importlib.invalidate_caches()
from ncs_toc import CheckConfig, run_review, save_results
from ncs_core import select_documents

print("검수 엔진 준비 완료")


## 3. 입력과 설정
`upload`: 원고와 개발계획서를 한 번에 선택합니다(내용으로 역할 판별).
`drive`: Google Drive를 마운트하고 `PATHS`에 경로를 입력합니다. `local`: 로컬 Jupyter 경로를 입력합니다.
파일명에 모듈 코드가 없거나 둘 이상이면 `MODULE_CODE`를 지정하세요. 역할 판별이 안 되면 `ROLES`를 지정하세요.


In [ ]:
INPUT_METHOD = "upload"  # upload / drive / local
PATHS = []  # drive·local: 원고와 개발계획서 경로. upload: 비워 두면 다음 셀에서 선택
ROLES = {}  # 역할 자동 판별이 안 될 때: {"원고.hwpx": "manuscript", "개발계획서.hwpx": "plan"}
MODULE_CODE = ""  # 예: "LM1203020201_24v3". 비우면 원고 파일명에서 추출
CONFIG = CheckConfig(front_section=0, body_section=1)
OUTPUT_DIR = "결과"


In [ ]:
result = saved = None
if INPUT_METHOD == "upload":
    from google.colab import files
    upload_root = Path(tempfile.mkdtemp(prefix="ncs_inputs_"))
    print("원고와 개발계획서를 선택하세요 (여러 개 선택 가능).")
    uploaded = files.upload()
    PATHS = []
    for filename, content in uploaded.items():
        path = upload_root / Path(filename).name
        path.write_bytes(content)
        PATHS.append(str(path))
elif INPUT_METHOD == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
elif INPUT_METHOD != "local":
    raise ValueError("INPUT_METHOD는 upload / drive / local 중 하나여야 합니다")
if not PATHS:
    raise ValueError("입력 파일이 없습니다. PATHS를 지정하거나 파일을 업로드하세요")
documents = select_documents(PATHS, ROLES, required=('manuscript',))
for role, document in documents.items():
    print(role, "→", document.path.name, f"({document.format}, 섹션 {len(document.sections)}개)")


## 4. 검사와 추출 결과 확인
먼저 진단의 `warnings`와 `counts`를 확인하세요.


In [ ]:
result = run_review(PATHS, roles=ROLES, module_code=MODULE_CODE, config=CONFIG)
print(json.dumps(result.diagnostics, ensure_ascii=False, indent=2))
print("읽은 차례:")
print(json.dumps(result.context.toc, ensure_ascii=False, indent=2))
print("본문 존재 근거:")
print(json.dumps(result.context.presence, ensure_ascii=False, indent=2))


## 5. 저장과 결과 보기
결과는 실행별 하위 폴더에 저장됩니다.


In [ ]:
if result is None:
    raise ValueError("검사 셀을 먼저 정상 완료하세요")
saved = save_results(result, OUTPUT_DIR)
print(json.dumps(saved.summary, ensure_ascii=False, indent=2))
print(f"지적사항 {len(result.findings)}건")
for finding in result.findings:
    print(finding["item_id"], "|", finding["check"], "|", finding["reason"])
try:
    import pandas as pd
    from IPython.display import display
except ImportError:
    print(json.dumps(result.checklist, ensure_ascii=False, indent=2))
else:
    rows = [{"항목": r["item_id"], "구역": r["region_path"], "판정": r["판정"], "체크": r["체크"], "비고": r["비고"]}
            for r in result.checklist]
    display(pd.DataFrame(rows))
print("결과 ZIP:", saved.zip_path)


In [ ]:
if saved is None:
    raise ValueError("저장 셀을 먼저 정상 완료하세요")
try:
    from google.colab import files
except ImportError:
    print("ZIP 위치:", saved.zip_path)
else:
    files.download(str(saved.zip_path))


## 팀 공유 시 읽는 순서
1. `ncs_toc/pipeline.py`: 검사 전체 흐름과 입력 오류 처리.
2. `ncs_toc/models.py`: 파서와 규칙이 주고받는 데이터.
3. `ncs_toc/parsers.py`: 문서에서 실제로 무엇을 읽었는지.
4. `ncs_toc/rules/`: 판정 기준. `ncs_toc/registry.py`: 새 규칙을 연결할 곳.
5. `ncs_core/`: 두 검수가 공유하는 문서 읽기(hwp/hwpx), 정규화·판정, 입력 역할 판별, 모듈 코드, 저장.

- 개발계획서 없이 실행하면 ③의 비교 기준이 차례가 되어 △(판단불가)가 늘어납니다.
- `차례_진단.json`의 `not_implemented`가 이 노트북이 검사하지 않는 항목입니다.
- `*_전체.json`은 모든 결과, `*_지적사항.json`은 X·△만, `*_진단.json`은 입력·읽기 상태를 담습니다.
- 결과는 실행마다 `결과/<모듈코드>_xxxx/` 하위 폴더에 저장되어 이전 결과를 덮어쓰지 않습니다.

페이지 좌표나 OCR을 사용하는 검사가 아니며, 이미지 속 글자를 읽지는 않습니다.
실제 개발계획서·원고 쌍으로 확인한 뒤 운영에 적용하세요.
